# 🚀 Notebook do Professor (Demo) — Aula 05: Embeddings e busca semântica com ChromaDB

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 05/14 — Módulo 2: RAG**  
**⏱️ 1h40min**  
**🔢 nomic-embed-text · ChromaDB**  
**🔁 Andaime 45%**  

---

## 🎯 Objetivo da aula

Entender como texto vira número e como números permitem encontrar documentos por significado — não por palavras-chave. Ao final, o grupo tem uma coleção ChromaDB do domínio pronta para o pipeline RAG da Aula 06.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz o lab do aluno com o gabarito das lacunas.

---

# 🔬 Código da aula — slide a slide

### Slide 08 — nomic-embed-text — setup no LangChain e Ollama

In [ ]:
!pip install langchain-ollama langchain-community chromadb -q

from langchain_ollama import OllamaEmbeddings
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# Modelo de embedding — não é um LLM de geração de texto
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",   # 274M params, janela 8k tokens
)

# Gerar embedding de uma frase — retorna lista de floats
vetor = embeddings.embed_query("Como fazer feijoada?")
print(f"Dimensões: {len(vetor)}")   # → 768 floats
print(f"Primeiros 5: {vetor[:5]}")  # → [-0.024, 0.031, ...]
print(f"Tipo: {type(vetor[0])}")    # → <class 'float'>

# Embeddings de múltiplos documentos de uma vez
docs = ["Pizza com queijo", "Macarrão ao sugo", "Motor de carro"]
vetores = embeddings.embed_documents(docs)
print(f"Documentos: {len(vetores)}")   # → 3
print(f"Dims cada: {len(vetores[0])}")  # → 768

### Slide 09 — Calcular similaridade cosseno — o mecanismo por baixo

In [ ]:
import numpy as np

def similaridade_cosseno(v1: list, v2: list) -> float:
    """Calcula similaridade entre dois vetores de embedding."""
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Comparar pares de frases
frases = {
    "pizza":    "Pizza napolitana com mozarela",
    "lasanha":  "Lasanha com queijo e presunto",
    "futebol":  "Futebol é o esporte mais popular",
    "carro":    "Motor do carro faz barulho",
}
vetores = {k: embeddings.embed_query(v) for k, v in frases.items()}

query_vec = embeddings.embed_query("comida italiana")

for nome, vec in vetores.items():
    sim = similaridade_cosseno(query_vec, vec)
    print(f"{nome:10s}: {sim:.4f}")
# pizza:    0.8945  ← alta similaridade
# lasanha:  0.8712  ← alta similaridade
# futebol:  0.4321  ← baixa similaridade
# carro:    0.2104  ← muito baixa similaridade

### Slide 11 — ChromaDB — add, query e persist

In [ ]:
import chromadb
from chromadb import Settings

# Cliente persistente — salva no disco (não perde ao reiniciar)
client = chromadb.PersistentClient(path="/content/chroma_db")

# Criar ou recuperar uma coleção
colecao = client.get_or_create_collection(
    name="dominio_grupo",
    metadata={"hnsw:space": "cosine"},  # usar similaridade cosseno
)

# Adicionar documentos com embeddings pré-computados
textos = ["Pizza napolitana com mozarela", "Lasanha bolonhesa com ricota"]
vetores = embeddings.embed_documents(textos)

colecao.add(
    documents=textos,
    embeddings=vetores,
    ids=["doc_1", "doc_2"],
    metadatas=[{"categoria":"italiana"}, {"categoria":"italiana"}],
)

# Buscar os k documentos mais similares à query
query_vec = embeddings.embed_query("comida italiana")
resultados = colecao.query(
    query_embeddings=[query_vec],
    n_results=2,
)
print(resultados["documents"])   # → [["Pizza napolitana...", "Lasanha..."]]
print(resultados["distances"])   # → [[0.10, 0.13]] (distância, não similaridade)

### Slide 12 — Metadata filtering — filtrar antes de buscar

In [ ]:
# Adicionar documentos com metadados ricos
colecao.add(
    documents=[
        "Pizza margherita é originária de Nápoles",
        "Frango à parmegiana — clássico brasileiro",
        "Churrasco gaúcho com costela e linguiça",
        "Feijoada completa com laranja",
    ],
    embeddings=embeddings.embed_documents([...]),
    ids=["d1","d2","d3","d4"],
    metadatas=[
        {"culinaria":"italiana",   "pais":"italia"},
        {"culinaria":"brasileira",  "pais":"brasil"},
        {"culinaria":"brasileira",  "pais":"brasil"},
        {"culinaria":"brasileira",  "pais":"brasil"},
    ],
)

# Buscar SOMENTE culinária brasileira
res = colecao.query(
    query_embeddings=[embeddings.embed_query("prato com carne")],
    n_results=2,
    where={"culinaria": "brasileira"},  # filtro antes da busca
)
# → pizza excluída mesmo que tenha boa similaridade com "carne"
# → retorna somente churrasco e feijoada

# Filtro com operador AND
colecao.query(..., where={"$and": [{"pais":"brasil"}, {"culinaria":"brasileira"}]})

### Slide 13 — ChromaDB via LangChain — a integração declarativa

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Criar vector store com LangChain Documents
documentos = [
    Document(page_content="Pizza napolitana", metadata={"fonte":"livro_1"}),
    Document(page_content="Macarrão ao sugo",   metadata={"fonte":"livro_1"}),
    Document(page_content="Churrasco gaúcho",   metadata={"fonte":"livro_2"}),
]

# from_documents calcula embeddings e salva tudo automaticamente
db = Chroma.from_documents(
    documents=documentos,
    embedding=embeddings,
    persist_directory="/content/chroma_langchain",
)

# Busca direta por similaridade
docs = db.similarity_search("comida italiana", k=2)
for d in docs:
    print(d.page_content, "|", d.metadata)

# Converter em retriever para uso em chain (Aula 06)
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},         # recuperar top-3
)
# → este retriever vai direto na chain RAG da Aula 06

### Slide 16 — Persistência e recarga da coleção

In [ ]:
# ─── SESSÃO 1: criar e salvar ───────────────────────────────
db = Chroma.from_documents(
    documents=documentos,
    embedding=embeddings,
    persist_directory="/content/chroma_ckp02",  # salva automaticamente
)
print(f"Documentos salvos: {db._collection.count()}")

# ─── SESSÃO 2: recarregar sem recalcular embeddings ─────────
db_recarregado = Chroma(
    persist_directory="/content/chroma_ckp02",  # lê do disco
    embedding_function=embeddings,              # para gerar embeddings de queries
)
print(f"Documentos recuperados: {db_recarregado._collection.count()}")

# Adicionar documentos a uma coleção existente
novos_docs = [Document(page_content="Novo documento do domínio")]
db_recarregado.add_documents(novos_docs)
# Só os novos docs têm embeddings computados — os antigos já estão no disco

# No Google Colab: copiar a pasta para o Drive para persistência entre sessões
!cp -r /content/chroma_ckp02 /content/drive/MyDrive/chroma_ckp02

### Slide 17 — similarity_search_with_score — avaliar a qualidade da recuperação

In [ ]:
# Busca com scores (distância cosseno — menor = mais similar)
resultados = db.similarity_search_with_score(
    "comida italiana",
    k=5,
)

for doc, score in resultados:
    # score é distância (0 = idêntico, 1 = sem relação)
    similaridade = 1 - score  # converter para similaridade
    relevante    = "✅" if similaridade > 0.75 else "⚠️"
    print(f"{relevante} [{similaridade:.3f}] {doc.page_content[:60]}")

# Exemplo de saída:
# ✅ [0.943] Pizza margherita com tomate fresco e mozarela
# ✅ [0.921] Lasanha bolonhesa com ricota cremosa
# ✅ [0.897] Macarrão al dente com pesto de manjericão
# ⚠️ [0.623] Frango à parmegiana — clássico brasileiro
# ⚠️ [0.412] Tacos com carne temperada

# Filtrar por threshold de relevância
THRESHOLD = 0.75
relevantes = [doc for doc, score in resultados if (1-score) > THRESHOLD]

### Slide 22 — Python novo desta aula

In [ ]:
import numpy as np
import uuid

# 1. numpy — dot product e norma vetorial
a = np.array([0.8, 0.3, 0.5])
b = np.array([0.7, 0.4, 0.6])
np.dot(a, b)           # produto escalar (dot product)
np.linalg.norm(a)      # norma (magnitude) do vetor

# 2. Dict comprehension com zip — gerar vetores por chave
nomes   = ["pizza", "carro"]
vetores = [embeddings.embed_query(n) for n in nomes]
mapa    = {k: v for k, v in zip(nomes, vetores)}  # zip e dict comp

# 3. uuid — gerar IDs únicos para documentos
doc_id = str(uuid.uuid4())   # "f47ac10b-58cc-4372-a567-0e02b2c3d479"

# 4. List comprehension com unpacking de tupla
resultados = [(doc, score)]  # retorno do similarity_search_with_score
relevantes = [doc for doc, score in resultados if (1-score) > 0.75]

# 5. f-string com formatação de float
print(f"Score: {score:.3f}")   # 3 casas decimais
print(f"Texto: {texto[:60]}")  # cortar string em 60 chars

---

# 💻 Lab do aluno — versão com lacunas

## 📋 Roteiro do Lab

**Lab — Aula 05 · 2º Semestre**  
### Coleção ChromaDB do domínio do grupo ★★

*Grupo 3–4 · 25 minutos · Google Colab*

1. Complete as 4 lacunas — modelo de embedding, 10+ documentos do domínio com metadados, criação do vector store e 5 queries semânticas.
2. Use metadados significativos para o domínio — ex: em culinária, uma categoria como "sobremesa"; em direito, a lei aplicável como "CDC". Esses metadados serão usados no RAG avançado da Aula 07.
3. Documente a qualidade de cada query num comentário: o resultado faz sentido? A similaridade reflete o esperado?
4. Desafio: calcule manualmente a similaridade cosseno entre 2 frases do corpus usando numpy (Slide 08) e compare com o score retornado pelo ChromaDB.

> **🎯 Gabarito das lacunas**
>
> Lacuna 1:  o nome do modelo de embedding usado no semestre inteiro — o mesmo configurado no Slide 07.
>
> Lacuna 2:  cada documento recebe o texto relevante do domínio do grupo e uma categoria como metadado — repetido para os 10+ documentos da lista.
>
> Lacuna 3:  o vector store recebe a lista de documentos e o objeto de embeddings criados nas lacunas anteriores.
>
> Lacuna 4:  5 frases que representam perguntas reais de alguém do domínio — quanto mais distantes das palavras exatas do corpus, melhor o teste de busca semântica.

> **💡 Dica de qualidade:**
>
> para testar se a busca está boa, use queries que não compartilham palavras com os documentos indexados. Se ainda retornar resultados relevantes, o embedding está funcionando bem para o domínio.

In [ ]:
!pip install langchain-ollama langchain-community chromadb numpy -q

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# 👉 LACUNA 1: instancie OllamaEmbeddings com o modelo correto
embeddings = OllamaEmbeddings(model=___)

# 👉 LACUNA 2: crie 10+ documentos do domínio do grupo
# Use Document(page_content="...", metadata={"categoria": "..."})
documentos = [
    Document(page_content=___, metadata={"categoria": ___}),
    # ... mais 9+ documentos
]

# 👉 LACUNA 3: crie o vector store com from_documents
db = Chroma.from_documents(
    documents=___,
    embedding=___,
    persist_directory="/content/chroma_ckp02",
)

# 👉 LACUNA 4: faça 5 queries semânticas e documente a qualidade
queries = [___, ___, ___, ___, ___]
for q in queries:
    res = db.similarity_search_with_score(q, k=3)
    print(f"\nQuery: {q}")
    for doc, score in res:
        print(f"  [{score:.3f}] {doc.page_content}")

# Converter em retriever para a Aula 06
retriever = db.as_retriever(search_kwargs={"k":3})
print(f"\nRetriever pronto para Aula 06: {type(retriever)}")

## 📚 Referências da aula

- Docs ChromaDB — Documentação oficial: collections, add, query, persist, metadata filtering. docs.trychroma.com
- Docs LangChain — OllamaEmbeddings e integração com ChromaDB. python.langchain.com/docs/integrations/vectorstores/chroma
- Modelo Nomic AI — nomic-embed-text: especificações, benchmarks e casos de uso. huggingface.co/nomic-ai/nomic-embed-text-v1
- Paper Mikolov, T. et al. — "Efficient Estimation of Word Representations in Vector Space." (Word2Vec, 2013) — a fundação conceitual dos embeddings modernos. arxiv.org/abs/1301.3781
- Livro Goodfellow, I.; Bengio, Y.; Courville, A. — Deep Learning. Pearson, 2017. Cap. 15: Representação distribuída — a base teórica dos embeddings e espaços vetoriais.

---

**→ Próxima Aula — Aula 06 · 14/09** — Pipeline RAG completo — load, split, embed, retrieve, generate
  
Conectar o retriever à chain LCEL. O LLM responde com base nos seus documentos.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*